# Подтяжка тарифа по спискам ИНН и договоров (с сохранением порядка)

Отдельная тетрадка для задачи:
- вход: два списка (ИНН и номера договоров),
- проверка 1:1 соответствия по порядку,
- поиск `agr_id` и `tariff_name` по нашему методу,
- итог строго в исходном порядке строк.

In [ ]:
import re
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 260)


def normalize_inn(v):
    if pd.isna(v):
        return None
    s = re.sub(r'\D+', '', str(v).strip())
    return s or None


def normalize_contract_key(v):
    if pd.isna(v):
        return None
    s = str(v).upper().replace('Ё', 'Е').strip()
    s = re.sub(r'\s+', '', s)
    s = re.sub(r'[^A-ZА-Я0-9]+', '', s)
    s = re.sub(r'^(?:ЕЭД|ЭДЭ|ВД|ЭД)', 'ЭД', s)
    return s or None


def contract_key_no_prefix(v):
    k = normalize_contract_key(v)
    if not k:
        return None
    return re.sub(r'^ЭД', '', k) or None


imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

print('Импала подключена')

In [ ]:
inn_block = '''
22904546495
22301387767
25901601810
20900006946
21402567115
23001394162
25400445422
190126797105
23600410381
24101691957
24000463496
26602829800
27003019911
24001819191
23203856814
22904546008
260995213
25405968578
21301709130
9101004689
27613526581
27607905782
23203251079
201014571
23501553406
278941762
278093664
271055194
26105049261
271007218
27207205031
274937275
24902888610
249008086
259013737
26302428773
21202662393
7401009490
24004005360
229955138
27311551302
26905782083
26408456410
26405808505
278984597
24900440156
26403057028
23401257160
26305481670
23002252957
381506530733
384201365512
382500009847
382500702120
850300920620
381298985920
3827024250
383225947229
380400493102
3811093837
3849108521
384200865978
3811148540
384200714496
850601033854
850500006558
383202869486
3814012402
382500774333
383600023036
4632235205
461300093462
462700260381
4634011240
462100377405
463226369757
4604005930
463207682210
461000126702
4615005441
4620003457
532204921221
532200066541
532200189705
5310015278
532200115340
5321128210
5304005560
5321026521
5316006209
530400175128
5302013069
532201079800
530200044343
5301004008
5321121944
543205523528
542206227826
5504216706
545105748909
222408337002
542908556606
544010981052
544606061745
5431074422
421711366819
5405510922
545114078390
5414101940
544616371432
5473021038
5436311356
542505297109
542708040726
540300690481
543672052340
5443026286
542212287407
543151118425
5418101645
5405116933
5440115989
542908004531
5445010059
543660146530
542206634758
542605055706
542709558750
545305119707
542510550364
543105433101
541003267740
542205068034
543151482713
544009586080
241101730060
542705867941
5434138762
561700288959
5629020603
5649001493
561200330578
560103880648
644007208342
5609051280
5614045115
561200923592
560504056798
5617021610
563802408271
564100003999
562400026390
560401899651
5641020568
5637981350
564203805419
564100018434
5637020364
564900085912
563400297868
565001777070
562800083876
561900228918
560204640032
563807536816
5638032281
561107013084
564100004230
564900024941
5601006510
565000068465
561200421257
562101214526
5836607990
731101997984
583514918695
583512881129
583500227835
583513742103
582800011916
583512672968
583700201288
583711805267
5813004378
582200186103
581801121701
583714156726
582900045228
581204004948
583600475790
582801514158
591402153401
312818884140
5914027910
5914026650
5948066386
591402371992
5914207448
5914021274
668506384079
665401813504
661219303387
667007346485
661505972406
665407140301
667000045143
664606142245
6657004010
665100973342
661106403870
661100070618
665500572401
660504107886
161405938623
163400006300
163401948977
165719940973
1612008209
161900156412
164413348221
1638001328
160902072445
161400165200
163501215204
165500015735
166009672915
1635004433
165207178350
162901167313
1647015729
160401206151
163800525927
161401690900
160907006291
1657054540
165206947934
1659104405
1660295264
'''

contract_block = '''
ЕЭД266219/00028
ЕЭД246233/00004
ЕЭД256223/00036
ЕЭД266213/00034
ЕЭД246242/00016
ЕЭД266206/00045
ЭД_Э-6203/20/23
ЕЭД266235/00033
ЕЭД246216/00011
ЕЭД246231/0009
ЕЭД246228/00014
ЕЭД266237/00015
ЕЭД256218/00027
ЕЭД256228/00018
ЕЭД256220/00035
ЭД_Э-6219/37/20
ЕЭД236211/0001
ЕЭД256203/00043
ЕЭД266236/00013
ЭД_300
ЭД_Э-6200/14/23
ЭД_Э-6235/110/16
ЕЭД246220/0013
ЕЭД246227/0017
ЕЭД246209/0016
ЭД_Э-6200/27/22
ЭД_Э-6200/81/18
ЭД_Э-6210/26/21
ЭД_Э-6213/64/17
ЭД_Э-6210/31/21
ЭД_Э-6221/12/22
ЕЭД266200/00010
ЭД_Э-6204/13/19
ЭД_Э-6204/12/20
ЭД_Э-6223/61/20
ЕЭД256209/00052
ЭД_Э-6234/19/19
ЭД_Э-6228/90/17
ЕЭД256228/00020
ЭД_Э-6219/02/23
ЭД_Э-6235/52/17
ЕЭД266205/00008
ЕЭД266267/00018
ЭД_Э-6219/24/21
ЕЭД246211/00008
ЕЭД266204/00002
ЭД_Э-6215/9/20
ЭД_Э-6229/8/18
ЕЭД256209/00055
ЭД_Э-6206/35/19
ЕЭД256609/00001
ЕЭД246603/0001
ЭД_000120
ЭД_000094
ЕЭД266607/00014
ЕЭД256615/00027
ЕЭД256600/00049
ЭД_000023
ЕЭД256610/00024
ЭД_000098
ЕЭД256601/00019
ЭД_000194
ЭД_000226
ЕЭД256603/00001
ЭД_000246
ЭД_000113
ЭД_000126
ЕЭД256613/00001
ЕЭД256613/00002
ЭД_000169
ЭД_148Т/3200
ЭД_14Т/3218
ЭД_8Т/3212
ЕЭД263226/00020
ЕЭД253207/00196
ЕЭД253200/00016
ЭД_03Т/3209
ЭД_108Т/3200
ЭД_11Т/3225
ЭД_9Т/3216
ЭД_7Т/3214
ЕЭД240800/00003
ЕЭД250804/00011
ЭД_170804/15
ЕЭД250800/00022
ЭД_170804/0112
ЭД_20221410
ЭД_20210414
ЭД_190800/2506
ЭД_190809/2506
ЭД_20210303
ЭД_170814/0919
ЕЭД260804/00002
ЭД_20221810
ЭД_08122020
ЭД_20211220
ЭД_162500/0008
ЕЭД262501/00037
ЕЭД262553/00005
ЕЭД262504/00044
ЕЭД262500/00043
ЕЭД262500/00027
ЕЭД252516/00051
ЭД_192500/154
ЭД_202500/192
ЕЭД242500/00037
ЕЭД262558/00038
ЕЭД262504/00007
ЭД_162500/028
ЭД_222500/237
ЕЭД252513/00025
ЕЭД242500/0007
ЭД_162500/010
ЭД_182500/115
ЕЭД252500/00042
ЭД_192500/150
ЭД_202500/180
ЭД_182500/103
ЕЭД252500/00007
ЕЭД242500/0010
ЭД_172500/089
ЭД_202500/201
ЭД_212500/222
ЭД_192500/175
ЭД_192500/141
ЭД_172500/041
ЭД_182500/127
ЭД_182500/125
ЭД_172500/058
ЭД_172500/071
ЭД_192500/173
ЕЭД242500/0008
ЭД_192500/159
ЭД_202500/203
ЭД_222500/248
ЕЭД252500/00010
ЭД_202500/200
ЕЭД242500/00024
ЭД_01-0511-21
ЕЭД260519/00077
ЕЭД250533/00001
ЕЭД260532/00003
ЕЭД240527/00002
ЕЭД240532/0002
ЕЭД250532/00001
ЕЭД250552/00100
ЭД_01-0509-21
ЕЭД240512/00001
ЕЭД230511/0001
ЭД_01-0532-19
ЭД_02-0514-23
ЕЭД250507/00001
ЕЭД230529/0001
ЕЭД250514/00040
ЭД_01-0509-18
ЕЭД260528/00018
ЭД_03-0514-17
ЭД_01-0509-22
ЭД_01-0533-20
ЕЭД240502/00001
ЭД_10-0504-17
02-0518-15
ЭД_01-0506-20
ЕЭД250507/00108
ЭД_02-0528-22
ЭД_01-0532-21
ЭД_02-0528-20
ЭД_05-0514-17
ЭД_06-0533-15
ЭД_01-0527-20
ЭД_04-0504-18
ЕЭД250528/00001
ЭД_02-0521-23
ЕЭД261500/00015
ЕЭД261510/00008
ЕЭД251500/00012
ЕЭД261500/00007
ЭД_80ТСП/1500
ЭД_175ТСП/1514
ЕЭД241513/0029
ЭД_68ТСП/1500
ЭД_183ТСП/1500
ЭД_015-35-21/65-2021
ЭД_109ТСП/1526
ЭД_015-35-21/90-2021
ЕЭД251515/00020
ЭД_91ТСП/1500
ЭД_172ТСП/1500
ЭД_137ТСП/1519
ЕЭД261500/00016
ЭД_РСХБ-015-35-21/7-2021
ЕЭД267605/00038
ЕЭД267605/00036
ЕЭД247600/0002
ЭД_44-2012/Д
ЕЭД267605/00043
ЕЭД257605/00008
ЭД_076/05/5-2023
ЕЭД257605/00002
ЕЭД257300/00050
ЕЭД267309/00035
ЕЭД257310/00046
ЕЭД267301/00003
ЕЭД247300/0018
ЕЭД257309/00020
ЕЭД267300/00047
ЕЭД267308/00017
ЭД_073-35-21/264
ЭД_073-35-21/262
ЕЭД267306/00038
ЭД_073-35-21/166
ЕЭД237300/0005
ЭД_073-35-21/177
ЕЭД246702/0008
ЭД_670083
ЭД_6700356
ЕЭД236703/0002
ЕЭД236703/0003
ЕЭД266700/00045
ЕЭД266721/00037
ЕЭД246700/00030
ЭД_6700181
ЭД_6700385
ЕЭД256700/00042
ЭД_6700390
ЭД_6700386
ЭД_6700209
ЕЭД256705/00004
ЭД_6700166
ЕЭД266716/00040
ЕЭД256700/00044
ЕЭД266719/00023
ЕЭД256702/00012
ЭД_6700330
ЭД_6700111
ЕЭД256705/00005
ЭД_6700377
ЭД_6700217
'''

inn_list = [x.strip() for x in inn_block.splitlines() if x.strip()]
contract_list = [x.strip() for x in contract_block.splitlines() if x.strip()]

print(f'ИНН: {len(inn_list):,}')
print(f'Договоров: {len(contract_list):,}')

if len(inn_list) != len(contract_list):
    raise ValueError(f'Количество ИНН ({len(inn_list)}) не равно количеству договоров ({len(contract_list)}).')

input_df = pd.DataFrame({
    'row_num': range(1, len(inn_list) + 1),
    'inn_input_raw': inn_list,
    'contract_input_raw': contract_list,
})

input_df['inn_key'] = input_df['inn_input_raw'].map(normalize_inn)
input_df['contract_key'] = input_df['contract_input_raw'].map(normalize_contract_key)
input_df['contract_key_np'] = input_df['contract_input_raw'].map(contract_key_no_prefix)

print('\nПервые 20 строк входа (порядок сохранен):')
display(input_df.head(20))

In [ ]:
inn_values = sorted([x for x in input_df['inn_key'].dropna().astype(str).unique().tolist() if x])
inn_sql_list = ', '.join([f"'{x}'" for x in inn_values]) if inn_values else "''"

sql_agreements = f"""
select
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.n_agr as string) as n_agr,
  cast(a.c_agr_number as string) as contract_number_alpha,
  cast(a.d_valid_from as date) as d_valid_from,
  cast(a.d_valid_to as date) as d_valid_to,
  upper(trim(cast(a.acq_class as string))) as acq_class,
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_key
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') in ({inn_sql_list})
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
"""

with imp:
    imp.execute('set MEM_LIMIT=8g')
    agreements_df = imp.fetch(sql_agreements)

if agreements_df is None:
    agreements_df = pd.DataFrame(columns=['agr_id', 'n_agr', 'contract_number_alpha', 'd_valid_from', 'd_valid_to', 'acq_class', 'inn_key'])

for c in ['agr_id', 'n_agr', 'contract_number_alpha', 'acq_class', 'inn_key']:
    if c in agreements_df.columns:
        agreements_df[c] = agreements_df[c].astype(str).str.strip()

agreements_df['contract_key'] = agreements_df['contract_number_alpha'].map(normalize_contract_key)
agreements_df['contract_key_np'] = agreements_df['contract_number_alpha'].map(contract_key_no_prefix)

# Кандидаты совпадений: по полному норм-ключу и по ключу без префикса ЭД
match_full_df = input_df.merge(
    agreements_df,
    on=['inn_key', 'contract_key'],
    how='left',
    suffixes=('_input', '')
)
match_full_df['match_rule'] = 'full_key'

match_np_df = input_df.merge(
    agreements_df,
    on=['inn_key', 'contract_key_np'],
    how='left',
    suffixes=('_input', '')
)
match_np_df['match_rule'] = 'no_prefix_key'

match_candidates_df = pd.concat([match_full_df, match_np_df], ignore_index=True)
match_candidates_df = match_candidates_df.drop_duplicates(
    subset=['row_num', 'inn_input_raw', 'contract_input_raw', 'agr_id', 'n_agr', 'contract_number_alpha']
)

matched_candidates_df = match_candidates_df[match_candidates_df['agr_id'].notna() & (match_candidates_df['agr_id'].astype(str).str.strip() != '')].copy()

if not matched_candidates_df.empty:
    matched_candidates_df['_d_valid_from_sort'] = pd.to_datetime(matched_candidates_df['d_valid_from'], errors='coerce')
    matched_best_df = (
        matched_candidates_df
        .sort_values(['row_num', '_d_valid_from_sort', 'n_agr'], ascending=[True, False, False])
        .drop_duplicates(subset=['row_num'], keep='first')
        .drop(columns=['_d_valid_from_sort'])
        .reset_index(drop=True)
    )
else:
    matched_best_df = pd.DataFrame(columns=list(input_df.columns) + ['agr_id', 'n_agr', 'contract_number_alpha', 'd_valid_from', 'd_valid_to', 'acq_class', 'match_rule'])

pair_df = input_df.merge(
    matched_best_df[[
        'row_num', 'agr_id', 'n_agr', 'contract_number_alpha', 'd_valid_from', 'd_valid_to', 'acq_class', 'match_rule'
    ]],
    on='row_num',
    how='left'
)

candidate_cnt_df = (
    matched_candidates_df.groupby('row_num', as_index=False)
    .agg(candidate_cnt=('agr_id', 'nunique'))
)
pair_df = pair_df.merge(candidate_cnt_df, on='row_num', how='left')
pair_df['candidate_cnt'] = pd.to_numeric(pair_df['candidate_cnt'], errors='coerce').fillna(0).astype(int)
pair_df['pair_status'] = pair_df['agr_id'].apply(lambda x: 'matched' if pd.notna(x) and str(x).strip() else 'no_agr_match')

print('QC по привязке ИНН+договор -> agr_id:')
print(f'Строк во входе: {len(input_df):,}')
print(f'Матчей agr_id: {(pair_df["pair_status"] == "matched").sum():,}')
print(f'Без agr_id: {(pair_df["pair_status"] == "no_agr_match").sum():,}')
print(f'Строк с несколькими кандидатами: {(pair_df["candidate_cnt"] > 1).sum():,}')

print('\nПервые 30 строк после привязки к agr_id:')
display(pair_df[['row_num', 'inn_input_raw', 'contract_input_raw', 'agr_id', 'contract_number_alpha', 'pair_status', 'candidate_cnt', 'match_rule']].head(30))

In [ ]:
agr_values = sorted([x for x in pair_df['agr_id'].dropna().astype(str).unique().tolist() if x])
agr_sql_list = ', '.join([f"'{x}'" for x in agr_values]) if agr_values else "''"

if agr_values:
    sql_tariff = f"""
    select
      cast(m.id as string) as agr_id,
      cast(m.c_tariff_plan as string) as c_tariff_plan,
      cast(tp.c_name as string) as tariff_name
    from ods.scd1_z_r2_ip_merchants m
    left join ods.scd1_z_r2_tariff_plan tp
      on cast(tp.id as string) = cast(m.c_tariff_plan as string)
    where cast(m.id as string) in ({agr_sql_list})
    """

    with imp:
        imp.execute('set MEM_LIMIT=8g')
        tariff_df = imp.fetch(sql_tariff)
else:
    tariff_df = pd.DataFrame(columns=['agr_id', 'c_tariff_plan', 'tariff_name'])

if tariff_df is None:
    tariff_df = pd.DataFrame(columns=['agr_id', 'c_tariff_plan', 'tariff_name'])

for c in ['agr_id', 'c_tariff_plan', 'tariff_name']:
    if c in tariff_df.columns:
        tariff_df[c] = tariff_df[c].astype(str).str.strip()

tariff_candidates_cnt_df = tariff_df.groupby('agr_id', as_index=False).agg(tariff_candidate_cnt=('c_tariff_plan', 'nunique')) if not tariff_df.empty else pd.DataFrame(columns=['agr_id', 'tariff_candidate_cnt'])

tariff_one_df = (
    tariff_df
    .sort_values(['agr_id', 'c_tariff_plan', 'tariff_name'], ascending=[True, True, True])
    .drop_duplicates(subset=['agr_id'], keep='first')
    .reset_index(drop=True)
) if not tariff_df.empty else pd.DataFrame(columns=['agr_id', 'c_tariff_plan', 'tariff_name'])

final_df = pair_df.merge(tariff_one_df, on='agr_id', how='left')
final_df = final_df.merge(tariff_candidates_cnt_df, on='agr_id', how='left')
final_df['tariff_candidate_cnt'] = pd.to_numeric(final_df['tariff_candidate_cnt'], errors='coerce').fillna(0).astype(int)

final_df['tariff_status'] = final_df['tariff_name'].apply(lambda x: 'tariff_found' if pd.notna(x) and str(x).strip() else 'tariff_not_found')
final_df['final_status'] = final_df.apply(
    lambda r: 'ok' if r['pair_status'] == 'matched' and r['tariff_status'] == 'tariff_found'
    else ('no_agr_match' if r['pair_status'] != 'matched' else 'no_tariff'),
    axis=1,
)

# Строго сохраняем исходный порядок
final_df = final_df.sort_values('row_num').reset_index(drop=True)

print('QC по итоговой подтяжке тарифа:')
print(f'Всего строк: {len(final_df):,}')
print(f'OK (agr_id + tariff): {(final_df["final_status"] == "ok").sum():,}')
print(f'Без agr_id: {(final_df["final_status"] == "no_agr_match").sum():,}')
print(f'Без tariff_name (при найденном agr_id): {(final_df["final_status"] == "no_tariff").sum():,}')
print(f'agr_id с >1 тарифным кандидатом: {(final_df["tariff_candidate_cnt"] > 1).sum():,}')

ordered_result_df = final_df[[
    'row_num',
    'inn_input_raw',
    'contract_input_raw',
    'agr_id',
    'contract_number_alpha',
    'tariff_name',
    'c_tariff_plan',
    'pair_status',
    'tariff_status',
    'final_status',
    'candidate_cnt',
    'tariff_candidate_cnt',
    'match_rule',
]].rename(columns={
    'inn_input_raw': 'inn',
    'contract_input_raw': 'contract_number_input',
    'contract_number_alpha': 'contract_number_alpha',
})

print('\nИтог (порядок как во входных списках):')
display(ordered_result_df.head(50))

if (ordered_result_df['final_status'] != 'ok').any():
    print('\nПроблемные строки:')
    display(ordered_result_df[ordered_result_df['final_status'] != 'ok'])

In [ ]:
# Опциональная выгрузка в Excel
output_path = Path('./inn_contract_tariff_ordered_result.xlsx')
ordered_result_df.to_excel(output_path, index=False)

print(f'Excel сохранен: {output_path.resolve()}')
print(f'Строк: {len(ordered_result_df):,}, колонок: {len(ordered_result_df.columns)}')